## 📊 Hackathon | Grupo 9 - Etapa: SQL na Prática
### Objetivo:

O objetivo desta etapa do projeto é aplicar SQL no tratamento e organização dos dados, visando estruturá-los de forma adequada para a extração de insights. A partir de queries, serão identificados padrões relacionados ao comportamento financeiro dos clientes e ao risco de inadimplência em empréstimos.

####1. Importação de bibliotecas

In [1]:
import sqlite3
import pandas as pd
from google.colab import files

####2. Upload do dataset

In [ ]:
uploaded = files.upload()

In [ ]:
df = pd.read_csv('loan_data_2015.csv')

/tmp/ipykernel_9243/3128102350.py:1: DtypeWarning: Columns (19,47,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('loan_data_2015.csv')


####3. Resumo dos dados

In [ ]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,60516983,64537751,20000,20000,20000,36 months,12.29,667.06,C,C1,...,NaN,NaN,NaN,NaN,NaN,NaN,41000,NaN,NaN,NaN
1,60187139,64163931,11000,11000,11000,36 months,12.69,369.00,C,C2,...,NaN,NaN,NaN,NaN,NaN,NaN,13100,NaN,NaN,NaN
2,60356453,64333218,7000,7000,7000,36 months,9.99,225.84,B,B3,...,NaN,NaN,NaN,NaN,NaN,NaN,16300,NaN,NaN,NaN
3,59955769,63900496,10000,10000,10000,36 months,10.99,327.34,B,B4,...,NaN,NaN,NaN,NaN,NaN,NaN,34750,NaN,NaN,NaN
4,58703693,62544456,9550,9550,9550,36 months,19.99,354.87,E,E4,...,NaN,NaN,NaN,NaN,NaN,NaN,14100,NaN,NaN,NaN


* O dataset contém 421.094 linhas e 74 colunas
* É composto por variáveis numéricas, categóricas e financeiras
* As principais colunas são:
  * `loan_amnt` → valor do empréstimo solicitado
  * `funded_amnt` → valor efetivamente financiado
  * `int_rate` → taxa de juros do empréstimo
  * `installment` → valor das parcelas
  * `annual_inc` → renda anual do cliente
  * `grade` e `sub_grade` → classificação de risco do crédito
  * `loan_status` → situação atual do empréstimo
  * `purpose` → finalidade do empréstimo
  * `home_ownership` → situação de moradia do cliente
  * `addr_state` → estado de residência do cliente
* O dataset apresenta valores nulos em algumas colunas e mistura de tipos de dados em alguns campos

####4. Integração do dataset ao SQLite

In [ ]:
conexao = sqlite3.connect('meu_banco.db')

df.to_sql(
    'loan_data_2015',
    conexao,
    if_exists='replace',
    index=False
)
print("Tudo certo!")

Tudo certo!


In [ ]:
cursor = conexao.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())

[('loan_data_2015',)]


####5. Validação da integração

In [ ]:
consulta = pd.read_sql_query("""
    SELECT *
    FROM loan_data_2015
    LIMIT 5
    """,
    conexao
)

consulta

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,60516983,64537751,20000,20000,20000,36 months,12.29,667.06,C,C1,...,None,None,None,None,None,None,41000,None,None,None
1,60187139,64163931,11000,11000,11000,36 months,12.69,369.00,C,C2,...,None,None,None,None,None,None,13100,None,None,None
2,60356453,64333218,7000,7000,7000,36 months,9.99,225.84,B,B3,...,None,None,None,None,None,None,16300,None,None,None
3,59955769,63900496,10000,10000,10000,36 months,10.99,327.34,B,B4,...,None,None,None,None,None,None,34750,None,None,None
4,58703693,62544456,9550,9550,9550,36 months,19.99,354.87,E,E4,...,None,None,None,None,None,None,14100,None,None,None


→ Visualização das cinco primeiras linhas

####6. Pré-processamento

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

consulta = pd.read_sql_query("""
    PRAGMA table_info(loan_data_2015)
""", conexao)

consulta

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,0
1,1,member_id,INTEGER,0,None,0
2,2,loan_amnt,INTEGER,0,None,0
3,3,funded_amnt,INTEGER,0,None,0
4,4,funded_amnt_inv,INTEGER,0,None,0
5,5,term,TEXT,0,None,0
6,6,int_rate,REAL,0,None,0
7,7,installment,REAL,0,None,0
8,8,grade,TEXT,0,None,0
9,9,sub_grade,TEXT,0,None,0


→ Conhecendo a estrutura das colunas
  * **INT**: 17 colunas
  * **FLOAT**: 34 colunas
  * **STR**: 23 colunas

In [ ]:
colunas = pd.read_sql_query("""
    PRAGMA table_info(loan_data_2015)
    """,
    conexao
)

query = "SELECT "

query += ", ".join([f"""
    SUM(
        CASE
            WHEN "{coluna}" IS NULL THEN 1
            ELSE 0
        END
    ) AS "{coluna}_nulos"
    """
    for coluna in colunas['name']
])

query += " FROM loan_data_2015"

consulta = pd.read_sql_query(
    query,
    conexao
)

with pd.option_context('display.max_columns', None):
    display(consulta)

,id_nulos,member_id_nulos,loan_amnt_nulos,funded_amnt_nulos,funded_amnt_inv_nulos,term_nulos,int_rate_nulos,installment_nulos,grade_nulos,sub_grade_nulos,emp_title_nulos,emp_length_nulos,home_ownership_nulos,annual_inc_nulos,verification_status_nulos,issue_d_nulos,loan_status_nulos,pymnt_plan_nulos,url_nulos,desc_nulos,purpose_nulos,title_nulos,zip_code_nulos,addr_state_nulos,dti_nulos,delinq_2yrs_nulos,earliest_cr_line_nulos,inq_last_6mths_nulos,mths_since_last_delinq_nulos,mths_since_last_record_nulos,open_acc_nulos,pub_rec_nulos,revol_bal_nulos,revol_util_nulos,total_acc_nulos,initial_list_status_nulos,out_prncp_nulos,out_prncp_inv_nulos,total_pymnt_nulos,total_pymnt_inv_nulos,total_rec_prncp_nulos,total_rec_int_nulos,total_rec_late_fee_nulos,recoveries_nulos,collection_recovery_fee_nulos,last_pymnt_d_nulos,last_pymnt_amnt_nulos,next_pymnt_d_nulos,last_credit_pull_d_nulos,collections_12_mths_ex_med_nulos,mths_since_last_major_derog_nulos,policy_code_nulos,application_type_nulos,annual_inc_joint_nulos,dti_joint_nulos,verification_status_joint_nulos,acc_now_delinq_nulos,tot_coll_amt_nulos,tot_cur_bal_nulos,open_acc_6m_nulos,open_il_6m_nulos,open_il_12m_nulos,open_il_24m_nulos,mths_since_rcnt_il_nulos,total_bal_il_nulos,il_util_nulos,open_rv_12m_nulos,open_rv_24m_nulos,max_bal_bc_nulos,all_util_nulos,total_rev_hi_lim_nulos,inq_fi_nulos,total_cu_tl_nulos,inq_last_12m_nulos
0,0,0,0,0,0,0,0,0,0,0,23874,23817,0,0,0,0,0,0,0,421049,0,132,0,0,0,0,0,0,203961,346679,0,0,0,162,0,0,0,0,0,0,0,0,0,0,0,17283,0,25757,11,0,298365,0,0,420583,420585,420583,0,0,0,399722,399722,399722,399722,400284,399722,402477,399722,399722,399722,399722,0,399722,399722,399722


→ Identificando colunas com dados nulos
  * **28 colunas**

In [ ]:
colunas = pd.read_sql_query("""
    PRAGMA table_info(loan_data_2015)
""", conexao)

total_linhas = pd.read_sql_query("""
    SELECT COUNT(*) AS total
    FROM loan_data_2015
""", conexao)["total"][0]

query = "SELECT "

query += ", ".join([f"""
    ROUND(
        SUM(CASE WHEN "{coluna}" IS NULL THEN 1 ELSE 0 END) * 100.0 / {total_linhas},
        2
    ) AS "{coluna}_perc_nulos"
"""
for coluna in colunas["name"]])

query += " FROM loan_data_2015"

consulta = pd.read_sql_query(query, conexao)

consulta.T.rename(columns={0: "percentual_nulos"}).sort_values(
    by="percentual_nulos",
    ascending=False
)

,percentual_nulos
desc_perc_nulos,99.99
annual_inc_joint_perc_nulos,99.88
dti_joint_perc_nulos,99.88
verification_status_joint_perc_nulos,99.88
il_util_perc_nulos,95.58
mths_since_rcnt_il_perc_nulos,95.06
open_rv_24m_perc_nulos,94.92
open_acc_6m_perc_nulos,94.92
all_util_perc_nulos,94.92
inq_last_12m_perc_nulos,94.92


→ Percentual de nulos por colunas

In [ ]:
resultado_nulos = consulta.T.rename(columns={0: "percentual_nulos"})

resultado_nulos[resultado_nulos["percentual_nulos"] >= 50].sort_values(
    by="percentual_nulos",
    ascending=False
)

,percentual_nulos
desc_perc_nulos,99.99
annual_inc_joint_perc_nulos,99.88
verification_status_joint_perc_nulos,99.88
dti_joint_perc_nulos,99.88
il_util_perc_nulos,95.58
mths_since_rcnt_il_perc_nulos,95.06
total_bal_il_perc_nulos,94.92
open_rv_12m_perc_nulos,94.92
open_il_6m_perc_nulos,94.92
open_il_12m_perc_nulos,94.92


→ Colunas com nulos e percentual

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        member_id,
        COUNT(*) AS quantidade
    FROM loan_data_2015
    GROUP BY member_id
    HAVING COUNT(*) > 1
    ORDER BY quantidade DESC
""", conexao)

consulta

,member_id,quantidade


→ Buscando `id` duplicados
  * **Não consta**

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        member_id,
        COUNT(*) AS quantidade
    FROM loan_data_2015
    GROUP BY member_id
    HAVING COUNT(*) > 1
    ORDER BY quantidade DESC
""", conexao)

consulta

,member_id,quantidade


→ Buscando `member_id` duplicados
  * **Não consta**

####7. Análise Exploratória de Dados



In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        loan_status,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM loan_data_2015), 2) AS percentual
    FROM loan_data_2015
    GROUP BY loan_status
    ORDER BY quantidade DESC
""", conexao)

consulta

,loan_status,quantidade,percentual
0,Current,377553,89.66
1,Fully Paid,22984,5.46
2,Issued,8460,2.01
3,Late (31-120 days),4691,1.11
4,In Grace Period,3107,0.74
5,Charged Off,2773,0.66
6,Late (16-30 days),1139,0.27
7,Default,387,0.09


→ Situação de quitação dos empréstimos

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        purpose,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM loan_data_2015), 2) AS percentual,
        ROUND(AVG(loan_amnt), 2) AS media_emprestimo,
        ROUND(AVG(dti), 2) AS media_dti
    FROM loan_data_2015
    GROUP BY purpose
    ORDER BY quantidade DESC
""", conexao)

consulta

,purpose,quantidade,percentual,media_emprestimo,media_dti
0,debt_consolidation,250020,59.37,15756.12,19.74
1,credit_card,102025,24.23,15951.62,19.26
2,home_improvement,25292,6.01,14750.91,16.69
3,other,19204,4.56,10345.20,18.20
4,major_purchase,7449,1.77,13058.91,15.89
5,medical,3938,0.94,9215.22,18.76
6,car,3466,0.82,10066.42,16.01
7,small_business,3364,0.80,15642.80,15.55
8,moving,2420,0.57,8314.12,17.44
9,vacation,2249,0.53,6524.46,18.41


→ Finalidade dos empréstimos cedidos


In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        grade,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM loan_data_2015), 2) AS percentual,
        ROUND(AVG(loan_amnt), 2) AS media_emprestimo,
        ROUND(AVG(int_rate), 2) AS media_taxa_juros,
        ROUND(AVG(annual_inc), 2) AS media_renda
    FROM loan_data_2015
    GROUP BY grade
    ORDER BY grade
""", conexao)

consulta

,grade,quantidade,percentual,media_emprestimo,media_taxa_juros,media_renda
0,A,73335,17.42,14691.72,6.94,91315.98
1,B,117606,27.93,14251.81,10.05,79193.29
2,C,120567,28.63,14745.59,13.30,72799.00
3,D,62654,14.88,15947.18,16.73,68462.79
4,E,34948,8.30,18472.73,19.29,70850.36
5,F,9817,2.33,20090.27,23.63,71779.30
6,G,2167,0.51,20427.84,26.84,70203.39


→ Distribuição por grade

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        home_ownership,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM loan_data_2015), 2) AS percentual
    FROM loan_data_2015
    GROUP BY home_ownership
    ORDER BY quantidade DESC
""", conexao)

consulta

,home_ownership,quantidade,percentual
0,MORTGAGE,207682,49.32
1,RENT,167644,39.81
2,OWN,45766,10.87
3,ANY,2,0.00


→ Regime de moradia dos clientes

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        MIN(loan_amnt) AS min_loan_amnt,
        MAX(loan_amnt) AS max_loan_amnt,
        ROUND(AVG(loan_amnt), 2) AS media_loan_amnt,

        MIN(annual_inc) AS min_annual_inc,
        MAX(annual_inc) AS max_annual_inc,
        ROUND(AVG(annual_inc), 2) AS media_annual_inc,

        MIN(dti) AS min_dti,
        MAX(dti) AS max_dti,
        ROUND(AVG(dti), 2) AS media_dti,

        MIN(int_rate) AS min_int_rate,
        MAX(int_rate) AS max_int_rate,
        ROUND(AVG(int_rate), 2) AS media_int_rate
    FROM loan_data_2015
""", conexao)

consulta

,min_loan_amnt,max_loan_amnt,media_loan_amnt,min_annual_inc,max_annual_inc,media_annual_inc,min_dti,max_dti,media_dti,min_int_rate,max_int_rate,media_int_rate
0,1000,35000,15240.26,0.0,9500000.0,76965.61,0.0,9999.0,19.2,5.32,28.99,12.6


→ Visão geral de valores mínimos, máximos e médias

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        issue_d,
        COUNT(*) AS quantidade,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM loan_data_2015), 2) AS percentual
    FROM loan_data_2015
    GROUP BY issue_d
    ORDER BY issue_d
""", conexao)

consulta

,issue_d,quantidade,percentual
0,Apr-15,35427,8.41
1,Aug-15,35886,8.52
2,Dec-15,44342,10.53
3,Feb-15,23770,5.64
4,Jan-15,35107,8.34
5,Jul-15,45962,10.91
6,Jun-15,28485,6.76
7,Mar-15,25400,6.03
8,May-15,31913,7.58
9,Nov-15,37530,8.91


→ Quantidade de empréstimos por mês de emissão

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        issue_d,
        loan_status,
        COUNT(*) AS quantidade
    FROM loan_data_2015
    GROUP BY issue_d, loan_status
    ORDER BY issue_d, quantidade DESC
""", conexao)

consulta

,issue_d,loan_status,quantidade
0,Apr-15,Current,30886
1,Apr-15,Fully Paid,2956
2,Apr-15,Late (31-120 days),648
3,Apr-15,Charged Off,428
4,Apr-15,In Grace Period,303
5,Apr-15,Late (16-30 days),144
6,Apr-15,Default,62
7,Aug-15,Current,33890
8,Aug-15,Fully Paid,1343
9,Aug-15,Late (31-120 days),299


→ Status dos empréstimos por mês

####8. Construção da tabela analítica

In [ ]:
# 1. Criação da tabela com os dados tratados
conexao.execute("""
    DROP TABLE IF EXISTS loan_data_2015_tratada
""")

conexao.execute("""
    CREATE TABLE loan_data_2015_tratada AS
    SELECT *
    FROM loan_data_2015
""")

conexao.commit()

In [ ]:
# 2. Confirmação de criação da tabela
consulta = pd.read_sql_query("""
    SELECT COUNT(*) AS total_linhas
    FROM loan_data_2015_tratada
""", conexao)

consulta

,total_linhas
0,421094


In [ ]:
# 3. Padronização das categorias
conexao.execute("""
    UPDATE loan_data_2015_tratada
    SET
        term = TRIM(term),
        grade = UPPER(TRIM(grade)),
        sub_grade = UPPER(TRIM(sub_grade)),
        home_ownership = UPPER(TRIM(home_ownership)),
        verification_status = TRIM(verification_status),
        loan_status = TRIM(loan_status),
        purpose = LOWER(TRIM(purpose))
""")

conexao.commit()

In [ ]:
# 4. Criação da coluna term_meses
conexao.execute("""
    ALTER TABLE loan_data_2015_tratada
    ADD COLUMN term_meses INTEGER
""")

conexao.execute("""
    UPDATE loan_data_2015_tratada
    SET term_meses = CAST(REPLACE(REPLACE(term, 'months', ''), ' ', '') AS INTEGER)
""")

conexao.commit()

In [ ]:
# 5. Criação da coluna emp_length_num

conexao.execute("""
    ALTER TABLE loan_data_2015_tratada
    ADD COLUMN emp_length_num INTEGER
""")

conexao.execute("""
    UPDATE loan_data_2015_tratada
    SET emp_length_num =
        CASE
            WHEN emp_length = '10+ years' THEN 10
            WHEN emp_length = '< 1 year' THEN 0
            WHEN emp_length IS NULL THEN NULL
            ELSE CAST(REPLACE(REPLACE(emp_length, ' years', ''), ' year', '') AS INTEGER)
        END
""")

conexao.commit()

In [ ]:
# 6. Criação da coluna de flag de inadimplência

conexao.execute("""
    ALTER TABLE loan_data_2015_tratada
    ADD COLUMN flag_inadimplente INTEGER
""")

conexao.execute("""
    UPDATE loan_data_2015_tratada
    SET flag_inadimplente =
        CASE
            WHEN loan_status IN (
                'Charged Off',
                'Default',
                'Late (31-120 days)',
                'Late (16-30 days)'
            ) THEN 1
            ELSE 0
        END
""")

conexao.commit()

In [ ]:
# 7. Criação da coluna de faixa de renda

conexao.execute("""
    ALTER TABLE loan_data_2015_tratada
    ADD COLUMN faixa_renda TEXT
""")

conexao.execute("""
    UPDATE loan_data_2015_tratada
    SET faixa_renda =
        CASE
            WHEN annual_inc IS NULL THEN 'Sem informação'
            WHEN annual_inc < 30000 THEN 'Baixa renda'
            WHEN annual_inc < 70000 THEN 'Média renda'
            WHEN annual_inc < 150000 THEN 'Alta renda'
            ELSE 'Renda muito alta'
        END
""")

conexao.commit()

In [ ]:
# 8. Criação da coluna razão empréstimo/renda

conexao.execute("""
    ALTER TABLE loan_data_2015_tratada
    ADD COLUMN razao_emprestimo_renda REAL
""")

conexao.execute("""
    UPDATE loan_data_2015_tratada
    SET razao_emprestimo_renda =
        CASE
            WHEN annual_inc IS NULL OR annual_inc = 0 THEN NULL
            ELSE ROUND(loan_amnt * 1.0 / annual_inc, 4)
        END
""")

conexao.commit()

In [ ]:
# 9. Visualização da tabela
consulta = pd.read_sql_query("""
    SELECT *
    FROM loan_data_2015_tratada
    LIMIT 10
""", conexao)

consulta

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_il_6m,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,term_meses,emp_length_num,flag_inadimplente,faixa_renda,razao_emprestimo_renda
0,60516983,64537751,20000,20000,20000,36 months,12.29,667.06,C,C1,Accounting Clerk,1 year,OWN,65000.0,Source Verified,Sep-15,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,None,debt_consolidation,Debt consolidation,542xx,WI,20.72,0,Sep-00,1,NaN,NaN,25,0,31578,77.0,42,w,0.0,0.0,0.00,0.00,0.00,0.00,0.0,0.0,0.0,None,0.00,None,Jan-16,0,NaN,1,INDIVIDUAL,None,None,None,0,0,52303,None,None,None,None,None,None,None,None,None,None,None,41000,None,None,None,36,1.0,1,Média renda,0.3077
1,60187139,64163931,11000,11000,11000,36 months,12.69,369.00,C,C2,Accounts Payable Lead,7 years,MORTGAGE,40000.0,Source Verified,Sep-15,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,None,debt_consolidation,Debt consolidation,235xx,VA,24.57,0,Sep-02,0,36.0,80.0,13,1,5084,38.8,41,w,0.0,0.0,10043.49,10043.49,9942.67,100.81,0.0,0.0,0.0,Oct-15,10059.00,None,Jan-16,0,79.0,1,INDIVIDUAL,None,None,None,0,332,175731,None,None,None,None,None,None,None,None,None,None,None,13100,None,None,None,36,7.0,1,Média renda,0.2750
2,60356453,64333218,7000,7000,7000,36 months,9.99,225.84,B,B3,Nurse,6 years,MORTGAGE,32000.0,Source Verified,Sep-15,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,None,debt_consolidation,Debt consolidation,350xx,AL,32.41,0,Feb-06,1,NaN,NaN,18,0,12070,74.0,36,f,0.0,0.0,221.96,221.96,167.56,54.40,0.0,0.0,0.0,Oct-15,225.84,None,Jan-16,0,NaN,1,INDIVIDUAL,None,None,None,0,0,202012,None,None,None,None,None,None,None,None,None,None,None,16300,None,None,None,36,6.0,1,Média renda,0.2188
3,59955769,63900496,10000,10000,10000,36 months,10.99,327.34,B,B4,Service Manager,10+ years,MORTGAGE,48000.0,Source Verified,Sep-15,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,None,credit_card,Credit card refinancing,483xx,MI,30.98,0,Oct-99,2,NaN,NaN,18,0,22950,66.0,41,f,0.0,0.0,315.13,315.13,235.76,79.37,0.0,0.0,0.0,Oct-15,327.34,None,Jan-16,0,NaN,1,INDIVIDUAL,None,None,None,0,0,108235,None,None,None,None,None,None,None,None,None,None,None,34750,None,None,None,36,10.0,1,Média renda,0.2083
4,58703693,62544456,9550,9550,9550,36 months,19.99,354.87,E,E4,None,None,RENT,32376.0,Verified,Sep-15,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,None,debt_consolidation,Debt consolidation,546xx,WI,32.54,0,Nov-99,3,69.0,NaN,9,0,4172,29.6,26,w,0.0,0.0,333.66,333.66,195.78,137.88,0.0,0.0,0.0,Oct-15,354.87,None,Jan-16,0,69.0,1,INDIVIDUAL,None,None,None,0,0,45492,None,None,None,None,None,None,None,None,None,None,None,14100,None,None,None,36,NaN,1,Média renda,0.2950
5,57783762,61536512,24000,24000,24000,60 months,14.65,566.56,C,C5,Owner,10+ years,MORTGAGE,70000.0,Not Verified,Aug-15,Charged Off,n,https://www.lendingclub.com/browse/loanDetail....,None,debt_consolidation,Debt consolidation,703xx,LA,6.96,0,May-98,0,65.0,NaN,8,0,8256,49.4,19,w,0.0,0.0,547.03,547.03,273.56,273.47,0.0,0.0,0.0,Oct-15,566.56,None,Jan-16,0,65.0,1,INDIVIDUAL,None,None,None,0,0,126165,None,None,None,None,None,None,None,None,N

####9. Elaboração de perguntas analíticas

######9.1 Quais faixas de risco (grade) possuem maior taxa de inadimplência?

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        grade,
        COUNT(*) AS total_clientes,

        SUM(
            CASE
                WHEN loan_status IN (
                    'Charged Off',
                    'Default',
                    'Late (31-120 days)',
                    'Late (16-30 days)'
                ) THEN 1
                ELSE 0
            END
        ) AS total_inadimplentes,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    GROUP BY grade
    ORDER BY taxa_inadimplencia DESC
""", conexao)

consulta

,grade,total_clientes,total_inadimplentes,taxa_inadimplencia
0,G,2167,233,10.75
1,F,9817,762,7.76
2,E,34948,1698,4.86
3,D,62654,2322,3.71
4,C,120567,2424,2.01
5,B,117606,1208,1.03
6,A,73335,343,0.47


**Descrição da análise**

Esta análise teve como objetivo investigar a relação entre a classificação de risco dos clientes (grade) e a taxa de inadimplência dos empréstimos. A intenção é identificar se clientes classificados em níveis de risco mais elevados apresentam maior probabilidade de atraso ou não pagamento, auxiliando na compreensão do comportamento de crédito e na identificação de perfis financeiros mais arriscados.<br><br>

**Interpretação dos resultados**

Os resultados demonstram uma relação clara entre o aumento do risco de crédito e o crescimento da inadimplência. Clientes classificados na categoria G, considerada a faixa de maior risco da base, apresentaram taxa de inadimplência de **`10,75%`**, enquanto clientes da categoria A, associada ao menor risco, registraram apenas **`0,47%`**.<br><br>

Observa-se também um crescimento progressivo da inadimplência conforme o risco aumenta:

**A**: 0,47%<br>
**B**: 1,03%<br>
**C**: 2,01%<br>
**D**: 3,71%<br>
**E**: 4,86%<br>
**F**: 7,76%<br>
**G**: 10,75%<br><br>

Esse comportamento indica que o sistema de classificação de risco da base apresenta forte coerência com o comportamento financeiro dos clientes, demonstrando potencial relevância para modelos preditivos de crédito e dashboards focados em alertas de risco financeiro.

######9.2 Clientes com maior relação empréstimo/renda possuem maior risco?

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        CASE
            WHEN annual_inc IS NULL THEN 'Sem informação'
            WHEN annual_inc < 30000 THEN 'Baixa renda'
            WHEN annual_inc < 70000 THEN 'Média renda'
            WHEN annual_inc < 150000 THEN 'Alta renda'
            ELSE 'Renda muito alta'
        END AS faixa_renda,

        ROUND(AVG(loan_amnt * 1.0 / annual_inc), 4) AS media_razao_emprestimo_renda,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    WHERE annual_inc IS NOT NULL
      AND annual_inc > 0
    GROUP BY faixa_renda
    ORDER BY media_razao_emprestimo_renda DESC
""", conexao)

consulta

,faixa_renda,media_razao_emprestimo_renda,taxa_inadimplencia
0,Baixa renda,0.2944,2.84
1,Média renda,0.2498,2.34
2,Alta renda,0.2015,1.86
3,Renda muito alta,0.1219,1.66


**Descrição da análise**

Esta análise buscou investigar a relação entre o valor do empréstimo em comparação à renda anual dos clientes e sua taxa de inadimplência. O objetivo é compreender se clientes mais comprometidos financeiramente apresentam maior risco de não pagamento, utilizando a razão entre empréstimo e renda como indicador de exposição financeira.<br><br>

**Interpretação dos resultados**

Os resultados sugerem uma relação inversa entre renda e risco financeiro. Clientes classificados na faixa de Baixa renda apresentaram a maior média de relação empréstimo/renda **`(0,2944)`**, além da maior taxa de inadimplência, com **`2,84%`**.<br><br>

À medida que a renda aumenta, observa-se redução tanto da relação empréstimo/renda quanto da inadimplência:<br><br>

**Baixa renda**: inadimplência de 2,84%<br>
**Média renda**: 2,34%<br>
**Alta renda**: 1,86%<br>
**Renda muito alta**: 1,66%<br><br>

Esse comportamento sugere que clientes com menor capacidade financeira tendem a assumir empréstimos proporcionalmente mais elevados em relação à própria renda, aumentando sua vulnerabilidade financeira e o risco de inadimplência.

######9.3 Quais finalidades de empréstimo apresentam maior inadimplência?

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        purpose,
        COUNT(*) AS total_clientes,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    GROUP BY purpose
    ORDER BY taxa_inadimplencia DESC
""", conexao)

consulta

,purpose,total_clientes,taxa_inadimplencia
0,wedding,4,25.00
1,small_business,3364,4.64
2,renewable_energy,224,4.46
3,moving,2420,4.09
4,house,1438,4.03
5,medical,3938,3.15
6,major_purchase,7449,2.78
7,other,19204,2.68
8,vacation,2249,2.67
9,debt_consolidation,250020,2.31


**Descrição da análise**

Esta análise teve como objetivo identificar quais finalidades de empréstimo (purpose) apresentam maiores taxas de inadimplência. A investigação busca compreender se determinados tipos de crédito estão associados a comportamentos financeiros mais arriscados, auxiliando na identificação de categorias com maior potencial de risco para instituições financeiras.<br><br>

**Interpretação dos resultados**

Os resultados indicam diferenças relevantes de inadimplência entre as finalidades dos empréstimos. A categoria small_business apresentou uma das maiores taxas de inadimplência entre categorias com volume significativo de clientes, alcançando **`4,64%`**, sugerindo maior risco associado a empréstimos voltados para pequenos negócios.<br><br>

Outras categorias com taxas elevadas incluem:

**renewable_energy**: 4,46%<br>
**moving**: 4,09%<br>
**house**: 4,03%<br>
**medical**: 3,15%<br><br>

Em contrapartida, categorias como credit_card apresentaram inadimplência mais baixa, com **`1,38%`**, mesmo possuindo alto volume de registros na base.<br><br>

A categoria wedding apresentou inadimplência de **`25%`**, porém possui apenas 4 registros, o que reduz sua relevância estatística e indica necessidade de cautela na interpretação.<br><br>

Os resultados demonstram que a finalidade do empréstimo pode representar um fator importante para análise de risco financeiro, contribuindo para modelos de score de crédito, segmentação de clientes e sistemas de alerta preventivo.

######9.4 Taxas de juros mais altas estão associadas a maior risco?

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        grade,

        ROUND(AVG(CAST(REPLACE(int_rate, '%', '') AS REAL)), 2) AS media_juros,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    GROUP BY grade
    ORDER BY media_juros DESC
""", conexao)

consulta

,grade,media_juros,taxa_inadimplencia
0,G,26.84,10.75
1,F,23.63,7.76
2,E,19.29,4.86
3,D,16.73,3.71
4,C,13.30,2.01
5,B,10.05,1.03
6,A,6.94,0.47


**Descrição da análise**

Esta análise teve como objetivo investigar a relação entre as taxas médias de juros aplicadas aos empréstimos e a taxa de inadimplência dos clientes. A proposta é compreender se clientes considerados mais arriscados financeiramente recebem juros maiores e se isso está associado ao aumento do risco de não pagamento.<br><br>

**Interpretação dos resultados**

Os resultados mostram uma forte relação entre aumento das taxas de juros e crescimento da inadimplência. Clientes classificados na categoria G, considerada a faixa de maior risco, apresentaram média de juros de **`26,84%`** e taxa de inadimplência de **`10,75%`**.<br><br>

Já clientes da categoria A, associados ao menor risco da base, registraram média de juros significativamente menor, de 6,94%, além de inadimplência de apenas **`0,47%`**.<br><br>

Observa-se um crescimento progressivo tanto dos juros quanto da inadimplência conforme o risco aumenta:

**A**: 6,94% de juros | 0,47% de inadimplência<br>
**B**: 10,05% | 1,03%<br>
**C**: 13,30% | 2,01%<br>
**D**: 16,73% | 3,71%<br>
**E**: 19,29% | 4,86%<br>
**F**: 23,63% | 7,76%<br>
**G**: 26,84% | 10,75%<br><br>

Esse comportamento indica que o sistema de concessão de crédito parece precificar corretamente o risco dos clientes, utilizando juros mais elevados para perfis com maior probabilidade de inadimplência.<br><br>

Os resultados também reforçam a relevância da taxa de juros como variável importante em modelos de score financeiro e análises de risco de crédito.

######9.5 Qual perfil de renda apresenta maior taxa de inadimplência?

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        CASE
            WHEN annual_inc IS NULL THEN 'Sem informação'
            WHEN annual_inc < 30000 THEN 'Baixa renda'
            WHEN annual_inc < 70000 THEN 'Média renda'
            WHEN annual_inc < 150000 THEN 'Alta renda'
            ELSE 'Renda muito alta'
        END AS faixa_renda,

        COUNT(*) AS total_clientes,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    GROUP BY faixa_renda
    ORDER BY taxa_inadimplencia DESC
""", conexao)

consulta

,faixa_renda,total_clientes,taxa_inadimplencia
0,Baixa renda,22959,2.84
1,Média renda,204811,2.34
2,Alta renda,166341,1.86
3,Renda muito alta,26983,1.66


**Descrição da análise**

Esta análise buscou identificar como diferentes faixas de renda se relacionam com o risco de inadimplência. O objetivo é compreender se clientes com menor capacidade financeira apresentam maior dificuldade de pagamento, contribuindo para análises de risco e segmentação financeira.<br><br>

**Interpretação dos resultados**

Os resultados indicam que clientes de menor renda apresentam maior taxa de inadimplência. A faixa classificada como Baixa renda registrou inadimplência de **`2,84%`**, representando o maior percentual entre os grupos analisados.<br><br>

À medida que a renda aumenta, observa-se redução gradual da inadimplência:

**Baixa renda**: 2,84%<br>
**Média renda**: 2,34%<br>
**Alta renda**: 1,86%<br>
**Renda muito alta**: 1,66%<br><br>

Além disso, a maior concentração de clientes encontra-se na faixa de Média renda, com 204.811 registros, seguida por Alta renda, com 166.341 clientes.<br><br>

Os resultados sugerem que clientes com maior capacidade financeira tendem a apresentar menor risco de inadimplência, reforçando a importância da renda como variável relevante para modelos de crédito, sistemas de score financeiro e mecanismos de alerta de risco.

######9.6 Clientes com maior DTI apresentam maior risco?

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        grade,

        ROUND(AVG(dti), 2) AS media_dti,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    GROUP BY grade
    ORDER BY media_dti DESC
""", conexao)

consulta

,grade,media_dti,taxa_inadimplencia
0,E,21.82,4.86
1,D,21.70,3.71
2,F,21.53,7.76
3,G,20.29,10.75
4,C,19.82,2.01
5,B,18.05,1.03
6,A,16.28,0.47


**Descrição da análise**

Esta análise teve como objetivo investigar a relação entre o índice DTI (Debt-to-Income Ratio), que representa o comprometimento da renda com dívidas, e a taxa de inadimplência dos clientes. A intenção é verificar se clientes com maior pressão financeira apresentam maior probabilidade de não pagamento.<br><br>

**Interpretação dos resultados**

Os resultados indicam que clientes classificados em categorias de maior risco apresentam médias de DTI mais elevadas e maiores taxas de inadimplência.<br><br>

As categorias E, D, F e G registraram os maiores níveis médios de DTI, variando entre aproximadamente 20 e 22 pontos, acompanhados por taxas mais elevadas de inadimplência.<br><br>

Destacam-se:

**G**: inadimplência de 10,75%
**F**: 7,76%
**E**: 4,86%
**D**: 3,71%<br><br>

Em contraste, clientes da categoria A apresentaram menor média de DTI **`(16,28)`** e inadimplência significativamente reduzida, de apenas **`0,47%`**.<br><br>

Os resultados sugerem que o comprometimento financeiro dos clientes possui relação relevante com o risco de crédito, indicando que o DTI pode representar uma variável importante em modelos preditivos de inadimplência e mecanismos de monitoramento financeiro.

######9.7 Quais clientes apresentam possíveis anomalias financeiras?

In [ ]:
consulta = pd.read_sql_query("""
    SELECT
        id,
        annual_inc,
        loan_amnt,
        dti,
        ROUND(loan_amnt * 1.0 / annual_inc, 4) AS razao_emprestimo_renda,
        grade,
        loan_status
    FROM loan_data_2015_tratada
    WHERE annual_inc IS NOT NULL
      AND annual_inc > 0
      AND (
            loan_amnt * 1.0 / annual_inc > 0.5
            OR dti > 30
            OR annual_inc < 25000
      )
    ORDER BY razao_emprestimo_renda DESC
    LIMIT 50
""", conexao)

consulta

,id,annual_inc,loan_amnt,dti,razao_emprestimo_renda,grade,loan_status
0,64078746,1200.00,12000,672.52,10.0000,E,Current
1,64957302,5000.00,19000,380.53,3.8000,D,Current
2,67405134,1770.00,6550,1092.52,3.7006,D,Current
3,66415388,8796.00,15675,34.52,1.7821,G,Current
4,65571637,8700.00,15000,120.66,1.7241,D,Current
5,63394642,12000.00,20000,68.30,1.6667,F,Current
6,67495417,17000.00,28000,136.97,1.6471,E,Issued
7,67467306,12000.00,18200,52.80,1.5167,E,Current
8,62245231,9000.00,12600,55.10,1.4000,E,Current
9,65067781,8160.00,10800,77.06,1.3235,E,Current


**Descrição da análise**

Esta análise teve como objetivo identificar possíveis anomalias financeiras na base de dados, considerando clientes com baixa renda anual, altos valores de empréstimo, elevado comprometimento financeiro (DTI) e alta relação entre empréstimo e renda.<br><br>

A proposta é localizar registros potencialmente fora do padrão esperado, que possam representar situações de risco elevado, inconsistências cadastrais ou necessidade de revisão manual por equipes de risco e compliance.<br><br>

**Interpretação dos resultados**

Os resultados identificaram diversos clientes com indicadores financeiros considerados extremamente elevados em relação à renda declarada.<br><br>

Alguns registros chamam atenção por apresentarem:

- relações empréstimo/renda superiores a 1,0;
- DTI extremamente elevado;
- empréstimos altos mesmo com baixa renda anual.<br><br>

O caso mais crítico encontrado apresentou:

- renda anual de apenas 1.200;
- empréstimo de 12.000;
- DTI de 672,52;
- relação empréstimo/renda de 10,0.<br><br>

Também foram encontrados clientes com:

- DTI acima de 1.000;
- empréstimos equivalentes ou superiores à renda anual;
- múltiplos registros classificados em categorias de risco D, E, F e G.<br><br>

Embora muitos desses empréstimos ainda estejam com status Current, os indicadores sugerem perfis potencialmente vulneráveis financeiramente ou situações que podem merecer investigação adicional.<br><br>

Esses resultados demonstram a importância de mecanismos de detecção de anomalias em sistemas financeiros, permitindo:

- geração de alertas automáticos;
- monitoramento preventivo de risco;
- apoio à equipe de compliance;
- identificação de possíveis inconsistências cadastrais ou concessões de crédito de alto risco.

######9.8 Existe relação entre tempo de emprego e inadimplência?

In [40]:
consulta = pd.read_sql_query("""
    SELECT
        emp_length,

        COUNT(*) AS total_clientes,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    WHERE emp_length IS NOT NULL

    GROUP BY emp_length
    ORDER BY taxa_inadimplencia DESC
""", conexao)

consulta

,emp_length,total_clientes,taxa_inadimplencia
0,< 1 year,34340,2.52
1,1 year,27473,2.51
2,6 years,16838,2.38
3,7 years,18414,2.36
4,4 years,24506,2.34
5,5 years,24930,2.25
6,2 years,37497,2.19
7,3 years,33430,2.17
8,9 years,16769,2.07
9,8 years,21560,1.89


**Descrição da análise**

Esta análise buscou investigar se o tempo de emprego dos clientes possui relação com a taxa de inadimplência. O objetivo é compreender se maior estabilidade profissional pode estar associada a menor risco financeiro.<br><br>

**Interpretação dos resultados**

Os resultados indicam uma tendência de redução da inadimplência conforme aumenta o tempo de emprego dos clientes.<br><br>

Clientes com menor tempo de experiência profissional apresentaram maiores taxas de inadimplência:

- **< 1 year: 2,52%**<br>
- **1 year: 2,51%**<br><br>

Já clientes com maior estabilidade profissional apresentaram menor risco:

- **10+ years: 1,80%**<br>
- **8 years: 1,89%**<br>
- **9 years: 2,07%**<br><br>

Além disso, a categoria 10+ years representa a maior concentração de clientes da base, com 141.520 registros.<br><br>

Os resultados sugerem que estabilidade profissional pode possuir relação relevante com capacidade de pagamento e risco financeiro, indicando potencial utilidade dessa variável em modelos de score de crédito e análises preditivas de inadimplência.

######9.9 Quais grades concentram os maiores empréstimos?

In [41]:
consulta = pd.read_sql_query("""
    SELECT
        grade,

        ROUND(AVG(loan_amnt), 2) AS media_emprestimo,

        MAX(loan_amnt) AS maior_emprestimo,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    GROUP BY grade
    ORDER BY media_emprestimo DESC
""", conexao)

consulta

,grade,media_emprestimo,maior_emprestimo,taxa_inadimplencia
0,G,20427.84,35000,10.75
1,F,20090.27,35000,7.76
2,E,18472.73,35000,4.86
3,D,15947.18,35000,3.71
4,C,14745.59,35000,2.01
5,A,14691.72,35000,0.47
6,B,14251.81,35000,1.03


**Descrição da análise**

Esta análise teve como objetivo investigar a relação entre as categorias de risco (grade), o valor médio dos empréstimos concedidos e a taxa de inadimplência dos clientes. A proposta é compreender se empréstimos mais elevados estão associados a perfis financeiros mais arriscados.<br><br>

**Interpretação dos resultados**

Os resultados mostram que clientes classificados em categorias de maior risco tendem a receber empréstimos médios mais elevados, além de apresentarem maiores taxas de inadimplência.<br><br>

A categoria G apresentou:

- **maior valor médio de empréstimo: 20.427,84**;
- **maior taxa de inadimplência: 10,75%**.<br><br>

As categorias F e E também registraram valores médios elevados:

- **F: 20.090,27 | inadimplência de 7,76%**
- **E: 18.472,73 | inadimplência de 4,86%**<br><br>

Em contrapartida, categorias de menor risco, como A e B, apresentaram menores médias de empréstimo e inadimplência significativamente reduzida.<br><br>

Observa-se ainda que o maior empréstimo registrado na base foi de 35.000, valor presente em todas as categorias analisadas.<br><br>

Os resultados sugerem que empréstimos de maior valor podem estar associados a níveis mais elevados de risco financeiro, reforçando a importância do monitoramento de grandes concessões de crédito em sistemas de análise de risco.

######9.10 Quais categorias possuem maior concentração de clientes de risco?

In [42]:
consulta = pd.read_sql_query("""
    SELECT
        home_ownership,

        COUNT(*) AS total_clientes,

        ROUND(
            SUM(
                CASE
                    WHEN loan_status IN (
                        'Charged Off',
                        'Default',
                        'Late (31-120 days)',
                        'Late (16-30 days)'
                    ) THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*),
            2
        ) AS taxa_inadimplencia

    FROM loan_data_2015_tratada
    GROUP BY home_ownership
    ORDER BY taxa_inadimplencia DESC
""", conexao)

consulta

,home_ownership,total_clientes,taxa_inadimplencia
0,RENT,167644,2.61
1,OWN,45766,2.21
2,MORTGAGE,207682,1.73
3,ANY,2,0.00


**Descrição da análise**

Esta análise teve como objetivo investigar a relação entre o tipo de moradia dos clientes (home_ownership) e a taxa de inadimplência dos empréstimos. A intenção é compreender se determinados perfis habitacionais apresentam maior vulnerabilidade financeira.<br><br>

**Interpretação dos resultados**

Os resultados indicam que clientes que vivem em imóveis alugados (RENT) apresentam a maior taxa de inadimplência da base, com **`2,61%`**.<br><br>

Em seguida aparecem:

- **OWN**: 2,21%
- **MORTGAGE**: 1,73%<br><br>

Além disso, clientes com financiamento imobiliário (MORTGAGE) representam a maior parcela da base, com 207.682 registros.<br><br>

Os resultados sugerem que clientes em situação de aluguel podem apresentar maior vulnerabilidade financeira em comparação a clientes com imóvel financiado ou próprio.<br><br>

A categoria ANY apresentou inadimplência de **`0%`**, porém possui apenas 2 registros, não sendo estatisticamente representativa para conclusões analíticas.<br><br>

Esses achados indicam que o perfil de moradia pode contribuir para análises de risco financeiro, segmentação de clientes e modelos preditivos de inadimplência.